In [ ]:
# ========================
# 07_metrics_to_semantic_with_skeleton.ipynb
# 從六大指標數據反過來生成 LLM 語義對齊文字，並在旁邊附加動態骨架以便對照
# ========================
import pandas as pd
import json
import numpy as np
import os
from pathlib import Path
from openai import OpenAI
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display, clear_output

# OpenAI API Key 設定
OPENAI_API_KEY = "sk-..."
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY.strip()
client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

print(f"OpenAI 模型已設定為：{OPENAI_MODEL}")

OpenAI 模型已設定為：gpt-4o-mini


In [2]:
# 定義分析資料夾路徑
folder = Path("C:/Users/AW'z/Downloads/ballet_Analysis_Results/data/streetdance-13_Analysis_Results/13-08/")

# 讀取指標資料與骨架資料
energy_df = pd.read_csv(folder / "energy.csv")
geometry_df = pd.read_csv(folder / "geometry.csv")
stability_df = pd.read_csv(folder / "stability.csv")
sync_df = pd.read_csv(folder / "synchronization.csv")
trans_df = pd.read_csv(folder / "transition.csv")
skeleton_df = pd.read_csv(folder / "13-08_skeleton.csv")

# 載入街舞文化資料庫 (Lookup Table)
cultural_lib_path = Path("C:/Users/AW'z/Downloads/ballet_Analysis_Results/street_dance_cultural_library.json")
with open(cultural_lib_path, "r", encoding="utf-8") as f:
    cultural_library = json.load(f)

print(f"✅ 指標與骨架資料載入成功！")
print(f"✅ 街舞文化資料庫載入成功！共 {len(cultural_library)} 筆項目")

✅ 指標與骨架資料載入成功！
✅ 街舞文化資料庫載入成功！共 175 筆項目


In [3]:
# 解析 skeleton.csv，將每幀的座標取出並轉換為 Numpy Array
num_frames = len(skeleton_df)
num_joints = 17
skel_data = np.zeros((num_frames, num_joints, 3))

for j in range(num_joints):
    col_str = skeleton_df[f'Joint_{j}']
    # 解析字串 'x, y, z' 到 float 陣列
    parsed = col_str.apply(lambda x: [float(v) for v in x.split(',')])
    skel_data[:, j, :] = np.vstack(parsed.values)

print(f"✅ 骨架資料解析完成！陣列形狀: {skel_data.shape} (Frames, Joints, XYZ)")

✅ 骨架資料解析完成！陣列形狀: (2248, 17, 3) (Frames, Joints, XYZ)


In [4]:
# 設定取樣間隔 (例如每 2 秒一個語義轉折點)
FPS = 30
INTERVAL_SEC = 2
INTERVAL_FRAMES = INTERVAL_SEC * FPS

total_frames = len(energy_df)
semantic_segments = []

for start_f in range(0, total_frames, INTERVAL_FRAMES):
    end_f = min(start_f + INTERVAL_FRAMES, total_frames)
    f_range = range(start_f, end_f)
    
    # 聚合這段時間的指標平均值
    seg_metrics = {
        'timestamp_sec': round(start_f / FPS, 2),
        'frame_start': start_f,
        'energy': energy_df.iloc[f_range]['energy'].mean(),
        'volume': geometry_df.iloc[f_range]['volume'].mean(),
        'curvature': geometry_df.iloc[f_range]['curvature'].mean(),
        'sway': stability_df.iloc[f_range]['sway'].mean(),
        'correlation': sync_df.iloc[f_range]['correlation'].mean(),
        'torque': trans_df.iloc[f_range]['torque'].mean(),
        'jerk': trans_df.iloc[f_range]['jerk'].mean()
    }
    semantic_segments.append(seg_metrics)

segments_df = pd.DataFrame(semantic_segments)
print(f"🔹 已切分為 {len(segments_df)} 個語義片段")

🔹 已切分為 38 個語義片段


In [5]:
# 將文化庫格式化為給 GPT 參考的字串
cultural_reference = "\n".join([f"- {item['description']}\n  (詩意詮釋: {item['poetic']})" for item in cultural_library])

SYSTEM_PROMPT = """
你是深耕街頭文化的「街舞導師與 Hip-Hop 文化專家」。
你擁有從歐美到亞洲各類街舞風格（Breaking, Popping, Locking, House等）的深厚知識。
我會給你一段時間內的舞姿物理指標數據。請根據這些數據「感知」舞者的動作動能，並對照下方的【街舞文化資料庫】給予回應。

回應格式：
【AI sees】
[基於數據描述當下的動作畫面。請將物理數據轉化為街舞專業術語或風格描述。
例如：如果能量極高且重心變化劇烈，可能是「Breaking 的 Power Move 旋轉」；
如果急動度(Jerk)高且肌肉收縮感強，可能是「Popping 的強力脈衝」；
如果節奏感強且穩定度高，可能是「Locking 的精準鎖定」。]

【AI says】
[以酷帥、專業、充滿街頭精神的語氣說一句話。請務必引用文化庫中的關鍵字、先驅人物或風格意義。]

【街舞文化資料庫參考】：
{} 
""".format(cultural_reference[:2500])

def generate_semantic_text(metrics):
    prompt = f"""
    當前舞姿指標：
    - 能量 (Energy): {metrics['energy']:.2f}
    - 體積 (Volume): {metrics['volume']:.4f}
    - 曲率 (Curvature): {metrics['curvature']:.2f}
    - 搖擺 (Sway): {metrics['sway']:.3f}
    - 肢體協調相關性 (Correlation): {metrics['correlation']:.3f}
    - 扭力 (Torque): {metrics['torque']:.2f}
    - 急動度 (Jerk): {metrics['jerk']:.2f}
    """
    
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content.strip()

print("LLM 生成邏輯準備完成！")

LLM 生成邏輯準備完成！


In [6]:
def create_skeleton_animation(skel_frames, fps=30):
    """將片段的 3D 骨架陣列繪製為動態對照的 HTML 影片"""
    fig = plt.figure(figsize=(4, 4))
    ax = fig.add_subplot(111, projection='3d')
    
    # 近似的 17 關節連線定義 (COCO/SMPL 風格)
    # 根據常見資料，若 0 是骨盆：
    bones = [
        (0, 1), (1, 2), (2, 3),        # 右腿
        (0, 4), (4, 5), (5, 6),        # 左腿
        (0, 7), (7, 8), (8, 9), (9, 10), # 軀幹與頭部
        (8, 11), (11, 12), (12, 13),   # 左手
        (8, 14), (14, 15), (15, 16)    # 右手
    ]
    
    lines = [ax.plot([], [], [], c='blue', lw=2)[0] for _ in bones]
    scat = ax.scatter([], [], [], c='red', s=20, alpha=0.5)
    
    # 設定適當的 3D 範圍
    ax.set_xlim([-1, 1])
    ax.set_ylim([-1, 1])
    ax.set_zlim([0, 2])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    # 設定視角
    ax.view_init(elev=10, azim=0)
    plt.close(fig) # 隱藏靜態圖表
    
    def update(frame_idx):
        pts = skel_frames[frame_idx]
        scat._offsets3d = (pts[:,0], pts[:,1], pts[:,2])
        for line, bone in zip(lines, bones):
            p1, p2 = pts[bone[0]], pts[bone[1]]
            line.set_data([p1[0], p2[0]], [p1[1], p2[1]])
            line.set_3d_properties([p1[2], p2[2]])
        return lines + [scat]
    
    anim = animation.FuncAnimation(fig, update, frames=len(skel_frames), interval=1000/fps, blit=False)
    return HTML(anim.to_jshtml())

print("骨架動畫繪製邏輯準備完成！")

骨架動畫繪製邏輯準備完成！


In [7]:
print("🚀 開始生成語義文字（這可能需要一些時間）...\n")

results = []
for i, row in segments_df.iterrows():
    print(f"正在處理片段 {i+1}/{len(segments_df)} (T={row['timestamp_sec']}s)...", end='\r')
    semantic_chat = generate_semantic_text(row)
    
    results.append({
        'timestamp': row['timestamp_sec'],
        'metrics': row.to_dict(),
        'llm_output': semantic_chat
    })

print("\n✨ 生成完成！")

🚀 開始生成語義文字（這可能需要一些時間）...

正在處理片段 38/38 (T=74.0s)...
✨ 生成完成！


In [8]:
for res in results[:5]:  # 顯示前 5 個結果作為範例
    print("=" * 60)
    print(f"時間: {res['timestamp']} 秒")
    print("-" * 30)
    print(res['llm_output'])
    
    # --- 新增的動態骨架對照 --- #
    start_f = int(res['metrics']['frame_start'])
    end_f = int(min(start_f + INTERVAL_FRAMES, num_frames))
    skel_frames = skel_data[start_f:end_f]
    
    if len(skel_frames) > 0:
        print("\n[動態骨架對照]:")
        anim_html = create_skeleton_animation(skel_frames, fps=FPS)
        display(anim_html)
    print() 

# 儲存結果
output_path = folder.parent / "semantic_alignment_from_metrics_with_skeleton.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n✅ 結果已儲存至: {output_path}")

時間: 0.0 秒
------------------------------
【AI sees】
這段舞姿展現出極高的能量與扭力，急動度的數據更是驚人，顯示舞者的動作急促且充滿力量。這樣的能量指數結合高曲率，讓我感受到一種正在進行的強烈的 Popping 藝術，舞者的肌肉快速收縮與放鬆，帶來如機械般的精準感。雖然肢體協調性略有負相關，但這樣的風格正是 Popping 的一部分，強調獨立的肢體動作與瞬間的震撼。這種急促的脈衝感也可以讓人聯想到 Breaking 中的 Power Moves，尤其是在強烈的扭力作用下，舞者似乎在不斷突破重力的束縛。

【AI says】
這是一場能量與扭力的狂歡，像是 Popin' Pete 在舞台上釋放的機械魅力，讓每一次的 pop 都在空氣中激起波紋，街頭的靈魂在這震撼的節奏中重生！

[動態骨架對照]:



時間: 2.0 秒
------------------------------
【AI sees】
根據你的數據，這位舞者展現出極高的能量(7.15)，結合強大的扭力(10.13)，可以感受到他正在進行一系列激烈的 Breaking 動作，可能是高難度的 Power Moves，如旋轉或翻轉。曲率(6.90)顯示出動作的流暢性，伴隨著急動度(82475.00)的急劇變化，強烈的肌肉收縮與放鬆感覺如同Popping的強力脈衝。雖然肢體協調相關性(0.108)較低，但這反而顯示出他在進行即興表演，運用動能與空間的自由探索，讓整體表現充滿不確定性與驚喜。

【AI says】
"在街頭的旋風中，他如同 Crazy Legs 般引領著每一個 Power Move，把重力的束縛撕裂，讓每一次的旋轉都成為對抗的宣言！"

[動態骨架對照]:



時間: 4.0 秒
------------------------------
【AI sees】
舞者的動作充滿了強烈的急動感，急動度(Jerk)極高，顯示出他在每一次動作中的瞬時變化與爆發力，彷彿是一位在街頭上展現著 Popping 精髓的舞者。能量值也相當不錯，雖然體積較小，但卻能夠通過急促的 muscle pops 和彈跳式的動作將空氣撕裂，呈現出令人驚嘆的視覺效果。曲率的高值則暗示著他在每一個動作中都有著流暢的連貫性，讓整體的表現如同波浪般起伏，帶著機械感與精準度。扭力指標的高數值則進一步強調了他在舞動過程中的力量，讓整體的風格更顯張揚與動感。

【AI says】
在這片街頭，他化身為一台精密的機器，Popping 的每一個脈動都是對節拍的狂熱回應，舞動中將每一個瞬間的力量轉化為無形的音樂，正如 Popin' Pete 所展現的那樣，讓每一次的 pop 都在空氣中回響！

[動態骨架對照]:



時間: 6.0 秒
------------------------------
【AI sees】
從你的舞姿數據來看，這位舞者展現出高度的急動度與扭力，並且在能量上達到了一個不錯的水平。這樣的數據暗示著一種強烈的 Popping 風格，尤其是肌肉的快速收縮與放鬆，讓動作充滿機械感和視覺衝擊力。肢體協調性較低，可能意味著舞者在即興創作中更偏向於表達情感，而非追求完美的流暢度，這讓整體動作顯得更加生動。高曲率及急動度的結合，讓每一個「pop」都像是轟鳴的震撼，讓觀眾的目光無法移開。

【AI says】
你的每一次 pop，都像是電流穿透空氣，讓街頭的節奏瞬間活起來！這就是 Hip-Hop 精神的展現，正如 Popin' Pete 所示範的，讓我們在每個瞬間都感受到機械與人性的完美交融！

[動態骨架對照]:



時間: 8.0 秒
------------------------------
【AI sees】
當前舞姿的能量指標為 4.11，顯示出舞者的動作充滿活力與激情。這樣的高能量配合著 7.21 的扭力，讓我想到了 Breaking 中的 Power Moves，舞者的身體似乎在挑戰重力，以旋轉和翻轉的方式展現極限的力量與控制。急動度的數值 50819.57 則顯示出舞者在動作上有著強烈的瞬間爆發感，這讓我聯想到 Popping 的強力脈衝，肌肉的收縮與放鬆在空中交織出一種機械般的美感。整體的肢體協調性 0.338 雖然稍顯分散，但這種獨特的風格反而增添了舞者的個人魅力，讓他在舞台上更具個性。

【AI says】
看來你正在用你的身體打破重力的束縛，像 Crazy Legs 一樣在地板上翻轉出自己的故事！這股力量與激情，正是街頭文化中最純粹的表現，讓每一次旋轉都成為對抗現實的宣言！

[動態骨架對照]:




✅ 結果已儲存至: C:\Users\AW'z\Downloads\ballet_Analysis_Results\data\streetdance-13_Analysis_Results\semantic_alignment_from_metrics_with_skeleton.json
